# Galaxy10 DECaLS — pipeline reprodutível para classificação morfológica

Este notebook implementa um fluxo reprodutível para classificação morfológica de galáxias a partir do conjunto público **Galaxy10 DECaLS** (Leung & Bovy; DOI: `10.5281/zenodo.10845026`). As imagens possuem três canais derivados das bandas `g`, `r` e `z`, e os rótulos discretos são lidos do campo `ans` do arquivo HDF5.

O protocolo experimental inclui: agrupamento das dez classes originais em três macroclasses, balanceamento por *random undersampling* sem reposição, particionamento estratificado treino/validação/teste (70/15/15), treinamento YOLOv8n-CLS com múltiplos orçamentos de épocas e sementes, seleção por validação, avaliação no conjunto de teste, análise em função do redshift, ablações de resolução e aumento de dados e comparação com arquiteturas de referência.

Os limites de redshift e a seleção de hiperparâmetros são determinados sem utilizar o conjunto de teste.


## 0. Dependências

Instale as dependências apenas quando necessário. As versões efetivamente utilizadas são registradas durante a execução.


In [ ]:
%pip install -q ultralytics h5py requests tqdm scikit-learn pandas matplotlib pillow joblib torch torchvision timm


## 1. Ambiente computacional e reprodutibilidade


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import h5py
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import sklearn
import torch
import torchvision
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=UserWarning)

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("scikit-learn:", sklearn.__version__)

try:
    import ultralytics
    print("Ultralytics:", ultralytics.__version__)
except Exception:
    print("Ultralytics: versão não identificada")

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuração experimental


In [ ]:
@dataclass(frozen=True)
class Config:
    # Diretórios
    base_dir: Path = Path.cwd() / "galaxy10_revision"
    dataset_filename: str = "Galaxy10_DECals_NoDuplicated.h5"

    # Fonte oficial
    dataset_url: str = (
        "https://zenodo.org/records/10845026/files/"
        "Galaxy10_DECals_NoDuplicated.h5?download=1"
    )
    expected_md5: str = "a920094d5e470e3705f4691c0d6dff54"

    # Amostra e divisão
    balance_seed: int = 42
    split_seed: int = 42
    train_fraction: float = 0.70
    val_fraction: float = 0.15
    test_fraction: float = 0.15

    # YOLO
    yolo_weights: str = "yolov8n-cls.pt"
    image_sizes: tuple[int, ...] = (128, 256)
    main_image_size: int = 128
    epoch_budgets: tuple[int, ...] = (30, 50, 100)
    seeds: tuple[int, ...] = (42, 123, 2026)
    batch_size: int = 16
    patience: int = 15
    optimizer: str = "AdamW"
    initial_lr: float = 1.0e-3
    weight_decay: float = 5.0e-4
    workers: int = 4

    # Execução
    device: str = "0" if torch.cuda.is_available() else "cpu"
    quick_test: bool = False

    # Blocos opcionais e computacionalmente caros
    run_epoch_sweep: bool = True
    run_resolution_ablation: bool = True
    run_augmentation_ablation: bool = True
    run_redshift_specialists: bool = True
    run_baselines: bool = True

    # Redshift
    minimum_per_class_per_split_for_specialist: int = 30

    # Bootstrap
    bootstrap_iterations: int = 1000
    bootstrap_seed: int = 2026

CFG = Config()

BASE_DIR = CFG.base_dir
DATA_DIR = BASE_DIR / "data"
DATASET_PATH = DATA_DIR / CFG.dataset_filename
IMAGE_DATASET_DIR = BASE_DIR / "dataset_balanced_3classes"
MANIFEST_DIR = BASE_DIR / "manifests"
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = BASE_DIR / "figures"
YOLO_RUNS_DIR = BASE_DIR / "yolo_runs"
REDSHIFT_DATASETS_DIR = BASE_DIR / "redshift_datasets"
BASELINE_RUNS_DIR = BASE_DIR / "baseline_runs"

for directory in [
    BASE_DIR, DATA_DIR, MANIFEST_DIR, RESULTS_DIR, FIGURES_DIR,
    YOLO_RUNS_DIR, REDSHIFT_DATASETS_DIR, BASELINE_RUNS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

if CFG.quick_test:
    EPOCH_BUDGETS = (1,)
    SEEDS = (42,)
else:
    EPOCH_BUDGETS = CFG.epoch_budgets
    SEEDS = CFG.seeds

print(json.dumps({
    **asdict(CFG),
    "base_dir": str(CFG.base_dir),
}, indent=2, default=str))

## 3. Taxonomia e definição das macroclasses

As dez classes do Galaxy10 DECaLS são agregadas em três macroclasses para o experimento de classificação:

| ID | Classe original | Macroclasse |
|---:|---|---|
| 0 | Disturbed Galaxies | disturbed_merging |
| 1 | Merging Galaxies | disturbed_merging |
| 2 | Round Smooth Galaxies | smooth |
| 3 | In-between Round Smooth Galaxies | smooth |
| 4 | Cigar Shaped Smooth Galaxies | smooth |
| 5 | Barred Spiral Galaxies | disk_spiral |
| 6 | Unbarred Tight Spiral Galaxies | disk_spiral |
| 7 | Unbarred Loose Spiral Galaxies | disk_spiral |
| 8 | Edge-on Galaxies without Bulge | disk_spiral |
| 9 | Edge-on Galaxies with Bulge | disk_spiral |

A macroclasse `disk_spiral` inclui sistemas *edge-on* e evita pressupor a identificação visual de braços espirais nesses objetos.


In [ ]:
ORIGINAL_CLASS_NAMES = {
    0: "Disturbed Galaxies",
    1: "Merging Galaxies",
    2: "Round Smooth Galaxies",
    3: "In-between Round Smooth Galaxies",
    4: "Cigar Shaped Smooth Galaxies",
    5: "Barred Spiral Galaxies",
    6: "Unbarred Tight Spiral Galaxies",
    7: "Unbarred Loose Spiral Galaxies",
    8: "Edge-on Galaxies without Bulge",
    9: "Edge-on Galaxies with Bulge",
}

MACROCLASS_MAP = {
    0: "disturbed_merging",
    1: "disturbed_merging",
    2: "smooth",
    3: "smooth",
    4: "smooth",
    5: "disk_spiral",
    6: "disk_spiral",
    7: "disk_spiral",
    8: "disk_spiral",
    9: "disk_spiral",
}

CLASS_ORDER = ["disturbed_merging", "smooth", "disk_spiral"]
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_ORDER)}
INDEX_TO_CLASS = {idx: name for name, idx in CLASS_TO_INDEX.items()}

## 4. Aquisição dos dados e verificação de integridade


In [ ]:
def file_md5(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_streaming(
    url: str,
    destination: Path,
    expected_md5: str | None = None,
    force: bool = False,
) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and not force:
        if expected_md5:
            current_md5 = file_md5(destination)
            if current_md5.lower() == expected_md5.lower():
                print("Arquivo existente e MD5 válido:", destination)
                return destination
            print("MD5 inválido; o arquivo será baixado novamente.")
            destination.unlink()
        else:
            print("Arquivo existente:", destination)
            return destination

    temporary = destination.with_suffix(destination.suffix + ".part")
    if temporary.exists():
        temporary.unlink()

    print("Baixando:", url)
    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with temporary.open("wb") as handle, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    progress.update(len(chunk))

    temporary.replace(destination)

    if expected_md5:
        current_md5 = file_md5(destination)
        if current_md5.lower() != expected_md5.lower():
            destination.unlink(missing_ok=True)
            raise RuntimeError(
                f"MD5 inválido: esperado {expected_md5}, obtido {current_md5}."
            )

    print("Download concluído:", destination)
    return destination


download_streaming(
    CFG.dataset_url,
    DATASET_PATH,
    expected_md5=CFG.expected_md5,
    force=False,
)

## 5. Leitura do HDF5 e construção do catálogo


In [ ]:
def read_optional_dataset(handle: h5py.File, key: str, length: int) -> np.ndarray:
    if key not in handle:
        return np.full(length, np.nan)
    values = np.asarray(handle[key])
    values = np.squeeze(values)
    if len(values) != length:
        raise ValueError(f"A chave {key!r} possui comprimento incompatível.")
    return values


with h5py.File(DATASET_PATH, "r") as h5:
    print("Chaves:", list(h5.keys()))

    if "images" not in h5 or "ans" not in h5:
        raise KeyError("O HDF5 precisa conter as chaves 'images' e 'ans'.")

    image_shape = h5["images"].shape
    labels = np.asarray(h5["ans"]).astype(int).reshape(-1)
    n_objects = len(labels)

    if image_shape[0] != n_objects:
        raise ValueError("Número de imagens e rótulos incompatível.")

    ra = read_optional_dataset(h5, "ra", n_objects)
    dec = read_optional_dataset(h5, "dec", n_objects)
    redshift = read_optional_dataset(h5, "redshift", n_objects)
    pxscale = read_optional_dataset(h5, "pxscale", n_objects)

print("Formato das imagens:", image_shape)
print("Número de objetos:", n_objects)
print("IDs encontrados:", sorted(np.unique(labels).tolist()))

unknown_labels = sorted(set(np.unique(labels)) - set(MACROCLASS_MAP))
if unknown_labels:
    raise ValueError(f"Rótulos desconhecidos encontrados: {unknown_labels}")

catalog = pd.DataFrame({
    "source_index": np.arange(n_objects, dtype=int),
    "original_class_id": labels,
    "original_class_name": [ORIGINAL_CLASS_NAMES[int(x)] for x in labels],
    "macroclass": [MACROCLASS_MAP[int(x)] for x in labels],
    "ra_deg": pd.to_numeric(pd.Series(ra), errors="coerce"),
    "dec_deg": pd.to_numeric(pd.Series(dec), errors="coerce"),
    "redshift": pd.to_numeric(pd.Series(redshift), errors="coerce"),
    "pxscale_arcsec_per_pixel": pd.to_numeric(pd.Series(pxscale), errors="coerce"),
})

catalog["valid_redshift"] = (
    np.isfinite(catalog["redshift"]) & (catalog["redshift"] >= 0)
)

catalog_path = MANIFEST_DIR / "catalog_all_objects.csv"
catalog.to_csv(catalog_path, index=False)

print("\nContagem por classe original:")
display(
    catalog.groupby(
        ["original_class_id", "original_class_name", "macroclass"],
        observed=True,
    ).size().rename("count").reset_index()
)

print("\nContagem por macroclasse:")
display(catalog["macroclass"].value_counts().reindex(CLASS_ORDER).rename("count"))

print("\nResumo de redshift válido:")
display(catalog.loc[catalog["valid_redshift"], "redshift"].describe())

print("Catálogo salvo em:", catalog_path)

## 6. Inspeção visual da amostra original


In [ ]:
def show_random_examples(
    h5_path: Path,
    dataframe: pd.DataFrame,
    n_per_class: int = 4,
    seed: int = 42,
) -> None:
    rng = np.random.default_rng(seed)
    selected_rows = []

    for class_name in CLASS_ORDER:
        subset = dataframe.loc[dataframe["macroclass"] == class_name]
        n = min(n_per_class, len(subset))
        positions = rng.choice(subset.index.to_numpy(), size=n, replace=False)
        selected_rows.extend(dataframe.loc[positions].to_dict("records"))

    ncols = n_per_class
    nrows = len(CLASS_ORDER)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(3 * ncols, 3 * nrows))
    axes = np.atleast_2d(axes)

    with h5py.File(h5_path, "r") as h5:
        for row_idx, class_name in enumerate(CLASS_ORDER):
            class_rows = [r for r in selected_rows if r["macroclass"] == class_name]
            for col_idx in range(ncols):
                ax = axes[row_idx, col_idx]
                ax.axis("off")
                if col_idx >= len(class_rows):
                    continue
                row = class_rows[col_idx]
                image = np.asarray(h5["images"][int(row["source_index"])])
                ax.imshow(image)
                ax.set_title(
                    f"{class_name}\nID {int(row['original_class_id'])}: "
                    f"{row['original_class_name']}",
                    fontsize=8,
                )

    plt.tight_layout()
    output = FIGURES_DIR / "audit_random_examples.png"
    plt.savefig(output, dpi=200, bbox_inches="tight")
    plt.show()
    print("Figura salva em:", output)


show_random_examples(DATASET_PATH, catalog, n_per_class=4, seed=CFG.balance_seed)

## 7. Balanceamento das macroclasses

Aplica-se *random undersampling* sem reposição, usando como tamanho-alvo a menor macroclasse.


In [ ]:
def balance_catalog(
    dataframe: pd.DataFrame,
    class_column: str,
    classes: list[str],
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    counts = dataframe[class_column].value_counts().reindex(classes)
    if counts.isna().any():
        raise ValueError("Uma ou mais classes não estão presentes.")

    target_count = int(counts.min())
    rng = np.random.default_rng(seed)

    selected_parts = []
    selected_indices: set[int] = set()

    for class_name in classes:
        subset = dataframe.loc[dataframe[class_column] == class_name]
        chosen = rng.choice(
            subset.index.to_numpy(),
            size=target_count,
            replace=False,
        )
        selected_parts.append(dataframe.loc[chosen].copy())
        selected_indices.update(int(x) for x in chosen)

    balanced = pd.concat(selected_parts, ignore_index=True)
    balanced = balanced.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    balanced["selected_for_balanced_sample"] = True

    rejected = dataframe.loc[
        ~dataframe.index.isin(selected_indices)
    ].copy().reset_index(drop=True)
    rejected["selected_for_balanced_sample"] = False

    return balanced, rejected


balanced_catalog, rejected_catalog = balance_catalog(
    catalog,
    class_column="macroclass",
    classes=CLASS_ORDER,
    seed=CFG.balance_seed,
)

print("Distribuição balanceada:")
display(
    balanced_catalog["macroclass"]
    .value_counts()
    .reindex(CLASS_ORDER)
    .rename("count")
)

print("Total balanceado:", len(balanced_catalog))
print("Objetos não selecionados:", len(rejected_catalog))

## 8. Particionamento estratificado treino/validação/teste


In [ ]:
def stratified_train_val_test_split(
    dataframe: pd.DataFrame,
    label_column: str,
    train_fraction: float,
    val_fraction: float,
    test_fraction: float,
    seed: int,
) -> pd.DataFrame:
    total = train_fraction + val_fraction + test_fraction
    if not np.isclose(total, 1.0):
        raise ValueError("As frações precisam somar 1.")

    train_df, temporary_df = train_test_split(
        dataframe,
        test_size=(1.0 - train_fraction),
        stratify=dataframe[label_column],
        random_state=seed,
    )

    relative_test_fraction = test_fraction / (val_fraction + test_fraction)

    val_df, test_df = train_test_split(
        temporary_df,
        test_size=relative_test_fraction,
        stratify=temporary_df[label_column],
        random_state=seed,
    )

    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"

    result = pd.concat([train_df, val_df, test_df], ignore_index=True)

    if result["source_index"].duplicated().any():
        raise RuntimeError("Há objetos repetidos entre os splits.")

    if set(result["source_index"]) != set(dataframe["source_index"]):
        raise RuntimeError("O split perdeu ou adicionou objetos.")

    return result.sample(frac=1.0, random_state=seed).reset_index(drop=True)


balanced_manifest = stratified_train_val_test_split(
    balanced_catalog,
    label_column="macroclass",
    train_fraction=CFG.train_fraction,
    val_fraction=CFG.val_fraction,
    test_fraction=CFG.test_fraction,
    seed=CFG.split_seed,
)

split_table = (
    balanced_manifest.groupby(["split", "macroclass"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=["train", "val", "test"], columns=CLASS_ORDER)
)

display(split_table)
display(split_table.assign(total=split_table.sum(axis=1)))

balanced_manifest_path = MANIFEST_DIR / "balanced_manifest.csv"
balanced_manifest.to_csv(balanced_manifest_path, index=False)

rejected_path = MANIFEST_DIR / "objects_not_selected_by_undersampling.csv"
rejected_catalog.to_csv(rejected_path, index=False)

print("Manifesto balanceado:", balanced_manifest_path)
print("Objetos não selecionados:", rejected_path)

## 9. Definição dos intervalos de redshift

Os limites são estimados exclusivamente no conjunto de treinamento a partir dos tercis \(Q_{33}\) e \(Q_{67}\), evitando transferência de informação dos conjuntos de validação e teste.


In [ ]:
def compute_redshift_boundaries(
    manifest: pd.DataFrame,
    split: str = "train",
) -> tuple[float, float]:
    values = manifest.loc[
        (manifest["split"] == split) & manifest["valid_redshift"],
        "redshift",
    ].dropna()

    if len(values) < 3:
        raise ValueError("Redshifts válidos insuficientes.")

    q33, q67 = values.quantile([1 / 3, 2 / 3]).to_numpy()
    if not np.isfinite(q33) or not np.isfinite(q67) or q33 >= q67:
        raise ValueError("Não foi possível construir tercis de redshift.")
    return float(q33), float(q67)


def assign_redshift_bin(
    redshift: pd.Series,
    q33: float,
    q67: float,
) -> pd.Series:
    conditions = [
        redshift.notna() & (redshift >= 0) & (redshift <= q33),
        redshift.notna() & (redshift > q33) & (redshift <= q67),
        redshift.notna() & (redshift > q67),
    ]
    labels = ["low_z", "mid_z", "high_z"]
    return pd.Series(
        np.select(conditions, labels, default="invalid_z"),
        index=redshift.index,
        dtype="object",
    )


Q33, Q67 = compute_redshift_boundaries(balanced_manifest, split="train")
balanced_manifest["redshift_bin"] = assign_redshift_bin(
    balanced_manifest["redshift"], Q33, Q67
)

balanced_manifest.to_csv(balanced_manifest_path, index=False)

redshift_config = {
    "q33_train": Q33,
    "q67_train": Q67,
    "definition": {
        "low_z": f"0 <= z <= {Q33:.8f}",
        "mid_z": f"{Q33:.8f} < z <= {Q67:.8f}",
        "high_z": f"z > {Q67:.8f}",
    },
}

with (RESULTS_DIR / "redshift_boundaries.json").open("w", encoding="utf-8") as handle:
    json.dump(redshift_config, handle, indent=2)

print(json.dumps(redshift_config, indent=2))

display(
    balanced_manifest.groupby(
        ["split", "redshift_bin", "macroclass"],
        observed=True,
    ).size().rename("count").reset_index()
)

## 10. Exportação das imagens e estrutura de dados para YOLO-CLS


In [ ]:
def reset_directory(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def image_output_path(
    dataset_root: Path,
    split: str,
    class_name: str,
    source_index: int,
) -> Path:
    return dataset_root / split / class_name / f"galaxy_{source_index:08d}.png"


def export_images_from_hdf5(
    h5_path: Path,
    manifest: pd.DataFrame,
    dataset_root: Path,
    reset: bool = True,
) -> pd.DataFrame:
    if reset:
        reset_directory(dataset_root)

    for split in ["train", "val", "test"]:
        for class_name in CLASS_ORDER:
            (dataset_root / split / class_name).mkdir(parents=True, exist_ok=True)

    output_manifest = manifest.copy()
    output_paths = []

    with h5py.File(h5_path, "r") as h5:
        images = h5["images"]

        for row in tqdm(
            output_manifest.itertuples(index=False),
            total=len(output_manifest),
            desc="Exportando PNGs",
        ):
            destination = image_output_path(
                dataset_root,
                row.split,
                row.macroclass,
                int(row.source_index),
            )
            image = np.asarray(images[int(row.source_index)])
            Image.fromarray(image).save(destination, optimize=True)
            output_paths.append(str(destination.resolve()))

    output_manifest["image_path"] = output_paths

    expected = len(output_manifest)
    actual = len(list(dataset_root.glob("*/*/*.png")))
    if actual != expected:
        raise RuntimeError(f"Esperadas {expected} imagens, encontradas {actual}.")

    return output_manifest


balanced_manifest = export_images_from_hdf5(
    DATASET_PATH,
    balanced_manifest,
    IMAGE_DATASET_DIR,
    reset=True,
)

balanced_manifest.to_csv(balanced_manifest_path, index=False)
print("Dataset criado em:", IMAGE_DATASET_DIR)

## 11. Métricas, bootstrap e visualizações


In [ ]:
def source_index_from_path(path: str | Path) -> int:
    """
    Extrai o índice original de nomes no formato galaxy_00001234.png.

    Esta função deve receber o caminho real exportado pelo notebook,
    não o caminho temporário retornado internamente pelo Ultralytics.
    """
    path = Path(path)
    stem = path.stem

    if not stem.startswith("galaxy_"):
        raise ValueError(
            f"Nome de arquivo inesperado: {path}. "
            "Esperado o formato galaxy_XXXXXXXX.png."
        )

    index_text = stem.removeprefix("galaxy_")
    if not index_text.isdigit():
        raise ValueError(
            f"Não foi possível extrair o índice original de: {path}"
        )

    return int(index_text)


def predict_image_folder(
    model_path: str | Path,
    split_dir: Path,
    image_size: int,
    device: str,
    batch_size: int,
) -> pd.DataFrame:
    """
    Executa predições preservando a correspondência exata entre cada
    resultado e o caminho original da imagem.

    O Ultralytics pode retornar result.path com nomes temporários como
    'image0.jpg'. Por isso, usamos image_paths como fonte de verdade.
    """
    image_paths = sorted(split_dir.glob("*/*.png"))
    if not image_paths:
        raise FileNotFoundError(
            f"Nenhuma imagem encontrada em {split_dir}."
        )

    model = YOLO(str(model_path))
    records = []

    predictions = model.predict(
        source=[str(path) for path in image_paths],
        imgsz=image_size,
        batch=batch_size,
        device=device,
        stream=True,
        verbose=False,
    )

    for original_path, result in tqdm(
        zip(image_paths, predictions),
        total=len(image_paths),
        desc=f"Predição {split_dir.name}",
    ):
        if result.probs is None:
            raise RuntimeError(
                "O resultado não contém probabilidades de classificação."
            )

        predicted_index = int(result.probs.top1)
        predicted_name = str(result.names[predicted_index])
        confidence = float(result.probs.top1conf.cpu().item())
        true_name = original_path.parent.name

        records.append({
            "source_index": source_index_from_path(original_path),
            "image_path": str(original_path.resolve()),
            "true_class": true_name,
            "predicted_class": predicted_name,
            "confidence": confidence,
        })

    if len(records) != len(image_paths):
        raise RuntimeError(
            f"Número de predições incompatível: "
            f"{len(records)} resultados para {len(image_paths)} imagens."
        )

    return pd.DataFrame(records)


def bootstrap_metric_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    metric_function,
    iterations: int,
    seed: int,
    alpha: float = 0.05,
) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    values = []

    for _ in range(iterations):
        idx = rng.integers(0, n, size=n)
        try:
            values.append(float(metric_function(y_true[idx], y_pred[idx])))
        except Exception:
            continue

    if not values:
        return np.nan, np.nan

    lower = float(np.quantile(values, alpha / 2))
    upper = float(np.quantile(values, 1 - alpha / 2))
    return lower, upper


def calculate_classification_metrics(
    predictions: pd.DataFrame,
    class_order: list[str],
    bootstrap_iterations: int = 0,
    bootstrap_seed: int = 42,
) -> tuple[dict[str, Any], pd.DataFrame, np.ndarray, np.ndarray]:
    y_true = predictions["true_class"].to_numpy()
    y_pred = predictions["predicted_class"].to_numpy()

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=class_order,
        zero_division=0,
    )

    per_class = pd.DataFrame({
        "class": class_order,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support.astype(int),
    })

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "n_objects": int(len(predictions)),
        "mean_confidence": float(predictions["confidence"].mean()),
    }

    if bootstrap_iterations > 0:
        metric_functions = {
            "accuracy": accuracy_score,
            "balanced_accuracy": balanced_accuracy_score,
            "macro_f1": lambda a, b: f1_score(a, b, average="macro", zero_division=0),
            "weighted_f1": lambda a, b: f1_score(a, b, average="weighted", zero_division=0),
        }
        for name, function in metric_functions.items():
            low, high = bootstrap_metric_ci(
                y_true,
                y_pred,
                function,
                iterations=bootstrap_iterations,
                seed=bootstrap_seed,
            )
            metrics[f"{name}_ci95_low"] = low
            metrics[f"{name}_ci95_high"] = high

    cm_absolute = confusion_matrix(y_true, y_pred, labels=class_order)
    cm_normalized = confusion_matrix(
        y_true,
        y_pred,
        labels=class_order,
        normalize="true",
    )

    return metrics, per_class, cm_absolute, cm_normalized


def plot_confusion_matrix(
    matrix: np.ndarray,
    class_order: list[str],
    title: str,
    output_path: Path,
    normalized: bool,
) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, interpolation="nearest")
    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(len(class_order)),
        yticks=np.arange(len(class_order)),
        xticklabels=class_order,
        yticklabels=class_order,
        xlabel="Classe predita",
        ylabel="Classe verdadeira",
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    threshold = np.nanmax(matrix) / 2 if matrix.size else 0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            text = f"{value:.2f}" if normalized else f"{int(value)}"
            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
            )

    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def save_evaluation_bundle(
    predictions: pd.DataFrame,
    prefix: str,
    output_dir: Path,
    bootstrap_iterations: int = 0,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics, per_class, cm_abs, cm_norm = calculate_classification_metrics(
        predictions,
        CLASS_ORDER,
        bootstrap_iterations=bootstrap_iterations,
        bootstrap_seed=CFG.bootstrap_seed,
    )

    predictions.to_csv(output_dir / f"{prefix}_predictions.csv", index=False)
    per_class.to_csv(output_dir / f"{prefix}_per_class_metrics.csv", index=False)
    pd.DataFrame([metrics]).to_csv(
        output_dir / f"{prefix}_global_metrics.csv",
        index=False,
    )
    np.savetxt(output_dir / f"{prefix}_confusion_absolute.csv", cm_abs, delimiter=",")
    np.savetxt(output_dir / f"{prefix}_confusion_normalized.csv", cm_norm, delimiter=",")

    plot_confusion_matrix(
        cm_abs,
        CLASS_ORDER,
        f"Matriz de confusão absoluta — {prefix}",
        output_dir / f"{prefix}_confusion_absolute.png",
        normalized=False,
    )
    plot_confusion_matrix(
        cm_norm,
        CLASS_ORDER,
        f"Matriz de confusão normalizada — {prefix}",
        output_dir / f"{prefix}_confusion_normalized.png",
        normalized=True,
    )

    display(pd.DataFrame([metrics]))
    display(per_class)
    return metrics

## 12. Configuração das transformações de aumento de dados


In [ ]:
AUGMENTATION_CONFIGS = {
    "physical_aug": {
        # A orientação no céu não define a classe morfológica.
        "degrees": 180.0,
        "fliplr": 0.5,
        "flipud": 0.5,
        "translate": 0.05,
        "scale": 0.10,
        # Perturbações fotométricas moderadas.
        "hsv_h": 0.0,
        "hsv_s": 0.10,
        "hsv_v": 0.20,
        # Desabilita políticas implícitas para manter o experimento rastreável.
        "auto_augment": None,
        "erasing": 0.0,
    },
    "no_aug": {
        "degrees": 0.0,
        "fliplr": 0.0,
        "flipud": 0.0,
        "translate": 0.0,
        "scale": 0.0,
        "hsv_h": 0.0,
        "hsv_s": 0.0,
        "hsv_v": 0.0,
        "auto_augment": None,
        "erasing": 0.0,
    },
}

## 13. Treinamento do classificador YOLOv8-CLS


In [ ]:
def train_yolo_classifier(
    dataset_dir: Path,
    run_name: str,
    epochs: int,
    seed: int,
    image_size: int,
    augmentation_name: str,
    project_dir: Path = YOLO_RUNS_DIR,
) -> dict[str, Any]:
    if augmentation_name not in AUGMENTATION_CONFIGS:
        raise KeyError(f"Aumentação desconhecida: {augmentation_name}")

    set_global_seed(seed)
    run_dir = project_dir / run_name
    best_model_path = run_dir / "weights" / "best.pt"

    if best_model_path.exists():
        print("Checkpoint existente; treinamento ignorado:", best_model_path)
        return {
            "run_name": run_name,
            "run_dir": str(run_dir),
            "best_model_path": str(best_model_path),
            "epochs_budget": epochs,
            "seed": seed,
            "image_size": image_size,
            "augmentation": augmentation_name,
            "training_seconds": np.nan,
            "reused_checkpoint": True,
        }

    model = YOLO(CFG.yolo_weights)
    augmentation = AUGMENTATION_CONFIGS[augmentation_name]

    started = time.perf_counter()
    model.train(
        data=str(dataset_dir),
        epochs=int(epochs),
        imgsz=int(image_size),
        batch=int(CFG.batch_size),
        device=CFG.device,
        workers=int(CFG.workers),
        project=str(project_dir),
        name=run_name,
        exist_ok=False,
        pretrained=True,
        optimizer=CFG.optimizer,
        lr0=float(CFG.initial_lr),
        weight_decay=float(CFG.weight_decay),
        cos_lr=True,
        patience=int(CFG.patience),
        seed=int(seed),
        deterministic=True,
        plots=True,
        verbose=True,
        **augmentation,
    )
    elapsed = time.perf_counter() - started

    if not best_model_path.exists():
        raise FileNotFoundError(f"Checkpoint não encontrado: {best_model_path}")

    return {
        "run_name": run_name,
        "run_dir": str(run_dir),
        "best_model_path": str(best_model_path),
        "epochs_budget": epochs,
        "seed": seed,
        "image_size": image_size,
        "augmentation": augmentation_name,
        "training_seconds": elapsed,
        "reused_checkpoint": False,
    }

## 14. Experimentos por orçamento de épocas e semente

São avaliados os orçamentos de 30, 50 e 100 épocas em três sementes aleatórias, mantendo fixos os *splits* e o protocolo de otimização. A configuração é selecionada exclusivamente pelo desempenho de validação, com *early stopping*.


In [ ]:
epoch_sweep_records = []

if CFG.run_epoch_sweep:
    for epochs in EPOCH_BUDGETS:
        for seed in SEEDS:
            run_name = f"yolov8n_epochs{epochs}_seed{seed}_img{CFG.main_image_size}_physical_aug"

            training_info = train_yolo_classifier(
                dataset_dir=IMAGE_DATASET_DIR,
                run_name=run_name,
                epochs=epochs,
                seed=seed,
                image_size=CFG.main_image_size,
                augmentation_name="physical_aug",
            )

            validation_predictions = predict_image_folder(
                model_path=training_info["best_model_path"],
                split_dir=IMAGE_DATASET_DIR / "val",
                image_size=CFG.main_image_size,
                device=CFG.device,
                batch_size=CFG.batch_size,
            )

            validation_metrics, _, _, _ = calculate_classification_metrics(
                validation_predictions,
                CLASS_ORDER,
                bootstrap_iterations=0,
            )

            record = {**training_info, **{
                f"val_{key}": value
                for key, value in validation_metrics.items()
            }}
            epoch_sweep_records.append(record)

            validation_predictions.to_csv(
                RESULTS_DIR / f"{run_name}_validation_predictions.csv",
                index=False,
            )

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    epoch_sweep_df = pd.DataFrame(epoch_sweep_records)
    epoch_sweep_df.to_csv(
        RESULTS_DIR / "yolo_epoch_sweep_all_runs.csv",
        index=False,
    )
else:
    sweep_path = RESULTS_DIR / "yolo_epoch_sweep_all_runs.csv"
    if not sweep_path.exists():
        raise FileNotFoundError(
            "A varredura está desabilitada e não há resultados salvos."
        )
    epoch_sweep_df = pd.read_csv(sweep_path)

display(
    epoch_sweep_df.sort_values(
        ["val_macro_f1", "val_balanced_accuracy", "training_seconds"],
        ascending=[False, False, True],
    )
)

## 15. Agregação dos resultados por orçamento de épocas


In [ ]:
epoch_summary = (
    epoch_sweep_df.groupby("epochs_budget", observed=True)
    .agg(
        runs=("run_name", "count"),
        macro_f1_mean=("val_macro_f1", "mean"),
        macro_f1_std=("val_macro_f1", "std"),
        balanced_accuracy_mean=("val_balanced_accuracy", "mean"),
        balanced_accuracy_std=("val_balanced_accuracy", "std"),
        accuracy_mean=("val_accuracy", "mean"),
        accuracy_std=("val_accuracy", "std"),
        training_seconds_mean=("training_seconds", "mean"),
    )
    .reset_index()
)

epoch_summary.to_csv(
    RESULTS_DIR / "yolo_epoch_sweep_summary.csv",
    index=False,
)
display(epoch_summary)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(
    epoch_summary["epochs_budget"],
    epoch_summary["macro_f1_mean"],
    yerr=epoch_summary["macro_f1_std"].fillna(0),
    marker="o",
    capsize=5,
)
ax.set_xlabel("Orçamento máximo de épocas")
ax.set_ylabel("Macro-F1 de validação")
ax.set_title("Desempenho em função do orçamento de treinamento")
ax.grid(alpha=0.3)
fig.tight_layout()
figure_path = FIGURES_DIR / "epoch_budget_macro_f1.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

## 16. Seleção da configuração pelo conjunto de validação


In [ ]:
best_row = (
    epoch_sweep_df.sort_values(
        ["val_macro_f1", "val_balanced_accuracy", "training_seconds"],
        ascending=[False, False, True],
        na_position="last",
    )
    .iloc[0]
)

BEST_MODEL_PATH = Path(best_row["best_model_path"])
BEST_EPOCH_BUDGET = int(best_row["epochs_budget"])
BEST_SEED = int(best_row["seed"])
BEST_IMAGE_SIZE = int(best_row["image_size"])

print("Melhor execução:")
display(best_row.to_frame("value"))
print("Checkpoint:", BEST_MODEL_PATH)

## 17. Avaliação do modelo selecionado no conjunto de teste


In [ ]:
test_predictions = predict_image_folder(
    model_path=BEST_MODEL_PATH,
    split_dir=IMAGE_DATASET_DIR / "test",
    image_size=BEST_IMAGE_SIZE,
    device=CFG.device,
    batch_size=CFG.batch_size,
)

test_predictions = test_predictions.merge(
    balanced_manifest[
        [
            "source_index",
            "original_class_id",
            "original_class_name",
            "macroclass",
            "ra_deg",
            "dec_deg",
            "redshift",
            "redshift_bin",
            "pxscale_arcsec_per_pixel",
        ]
    ],
    on="source_index",
    how="left",
    validate="one_to_one",
)

test_metrics = save_evaluation_bundle(
    test_predictions,
    prefix="best_yolo_test",
    output_dir=RESULTS_DIR / "best_yolo_test",
    bootstrap_iterations=CFG.bootstrap_iterations,
)

## 18. Desempenho do modelo em função do redshift


In [ ]:
def evaluate_by_group(
    predictions: pd.DataFrame,
    group_column: str,
    valid_groups: list[str],
    output_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    global_records = []
    class_records = []

    for group_name in valid_groups:
        subset = predictions.loc[predictions[group_column] == group_name].copy()
        if subset.empty:
            continue

        metrics, per_class, _, _ = calculate_classification_metrics(
            subset,
            CLASS_ORDER,
            bootstrap_iterations=CFG.bootstrap_iterations,
            bootstrap_seed=CFG.bootstrap_seed,
        )
        metrics[group_column] = group_name
        global_records.append(metrics)

        per_class[group_column] = group_name
        class_records.append(per_class)

    global_df = pd.DataFrame(global_records)
    class_df = pd.concat(class_records, ignore_index=True) if class_records else pd.DataFrame()

    global_df.to_csv(output_path.with_name(output_path.stem + "_global.csv"), index=False)
    class_df.to_csv(output_path.with_name(output_path.stem + "_per_class.csv"), index=False)

    return global_df, class_df


redshift_global_metrics, redshift_class_metrics = evaluate_by_group(
    test_predictions,
    group_column="redshift_bin",
    valid_groups=["low_z", "mid_z", "high_z"],
    output_path=RESULTS_DIR / "global_model_by_redshift.csv",
)

display(redshift_global_metrics)
display(redshift_class_metrics)

if not redshift_global_metrics.empty:
    order = ["low_z", "mid_z", "high_z"]
    plotting = (
        redshift_global_metrics.set_index("redshift_bin")
        .reindex(order)
        .dropna(subset=["macro_f1"])
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(plotting.index, plotting["macro_f1"], marker="o", label="Macro-F1")
    ax.plot(
        plotting.index,
        plotting["balanced_accuracy"],
        marker="s",
        label="Balanced accuracy",
    )
    ax.set_xlabel("Faixa de redshift")
    ax.set_ylabel("Métrica")
    ax.set_ylim(0, 1)
    ax.set_title("Desempenho do modelo global por redshift")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / "global_model_performance_by_redshift.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

## 19. Modelos especializados por intervalo de redshift

Para cada intervalo, preservam-se os *splits* globais e aplica-se balanceamento interno por *undersampling*. O treinamento é realizado apenas quando todas as macroclasses atingem o suporte mínimo definido na configuração.


In [ ]:
def link_or_copy(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(source, destination)
    except (OSError, AttributeError):
        shutil.copy2(source, destination)


def build_balanced_redshift_dataset(
    manifest: pd.DataFrame,
    redshift_bin: str,
    output_dir: Path,
    seed: int,
    minimum_per_class: int,
) -> tuple[Path, pd.DataFrame] | None:
    subset = manifest.loc[
        manifest["redshift_bin"] == redshift_bin
    ].copy()

    if subset.empty:
        return None

    rng = np.random.default_rng(seed)
    selected_parts = []

    for split in ["train", "val", "test"]:
        split_df = subset.loc[subset["split"] == split]
        counts = split_df["macroclass"].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
        target = int(counts.min())

        if target < minimum_per_class:
            print(
                f"{redshift_bin}/{split}: suporte mínimo {target} "
                f"< {minimum_per_class}; especialista ignorado."
            )
            return None

        for class_name in CLASS_ORDER:
            class_df = split_df.loc[split_df["macroclass"] == class_name]
            selected_index = rng.choice(
                class_df.index.to_numpy(),
                size=target,
                replace=False,
            )
            selected_parts.append(class_df.loc[selected_index].copy())

    selected = pd.concat(selected_parts, ignore_index=True)

    reset_directory(output_dir)
    for row in tqdm(
        selected.itertuples(index=False),
        total=len(selected),
        desc=f"Dataset {redshift_bin}",
    ):
        source = Path(row.image_path)
        destination = (
            output_dir
            / row.split
            / row.macroclass
            / source.name
        )
        link_or_copy(source, destination)

    selected.to_csv(output_dir / "manifest.csv", index=False)
    return output_dir, selected


redshift_specialist_records = []

if CFG.run_redshift_specialists:
    for redshift_bin in ["low_z", "mid_z", "high_z"]:
        specialist_dataset_dir = REDSHIFT_DATASETS_DIR / redshift_bin

        prepared = build_balanced_redshift_dataset(
            balanced_manifest,
            redshift_bin=redshift_bin,
            output_dir=specialist_dataset_dir,
            seed=CFG.split_seed,
            minimum_per_class=CFG.minimum_per_class_per_split_for_specialist,
        )

        if prepared is None:
            continue

        _, specialist_manifest = prepared

        for seed in SEEDS:
            run_name = (
                f"specialist_{redshift_bin}_epochs{BEST_EPOCH_BUDGET}_"
                f"seed{seed}_img{BEST_IMAGE_SIZE}"
            )

            training_info = train_yolo_classifier(
                dataset_dir=specialist_dataset_dir,
                run_name=run_name,
                epochs=BEST_EPOCH_BUDGET,
                seed=seed,
                image_size=BEST_IMAGE_SIZE,
                augmentation_name="physical_aug",
                project_dir=YOLO_RUNS_DIR / "redshift_specialists",
            )

            predictions = predict_image_folder(
                model_path=training_info["best_model_path"],
                split_dir=specialist_dataset_dir / "test",
                image_size=BEST_IMAGE_SIZE,
                device=CFG.device,
                batch_size=CFG.batch_size,
            )

            metrics, _, _, _ = calculate_classification_metrics(
                predictions,
                CLASS_ORDER,
                bootstrap_iterations=0,
            )

            redshift_specialist_records.append({
                **training_info,
                "redshift_bin": redshift_bin,
                **{f"test_{k}": v for k, v in metrics.items()},
            })

    redshift_specialist_df = pd.DataFrame(redshift_specialist_records)
    redshift_specialist_df.to_csv(
        RESULTS_DIR / "redshift_specialist_models.csv",
        index=False,
    )
    display(redshift_specialist_df)
else:
    print("Modelos especializados por redshift desabilitados.")

## 20. Ablação de resolução espacial


In [ ]:
resolution_records = []

if CFG.run_resolution_ablation:
    for image_size in CFG.image_sizes:
        for seed in SEEDS:
            run_name = (
                f"resolution_img{image_size}_epochs{BEST_EPOCH_BUDGET}_"
                f"seed{seed}_physical_aug"
            )

            training_info = train_yolo_classifier(
                dataset_dir=IMAGE_DATASET_DIR,
                run_name=run_name,
                epochs=BEST_EPOCH_BUDGET,
                seed=seed,
                image_size=image_size,
                augmentation_name="physical_aug",
                project_dir=YOLO_RUNS_DIR / "resolution_ablation",
            )

            predictions = predict_image_folder(
                model_path=training_info["best_model_path"],
                split_dir=IMAGE_DATASET_DIR / "val",
                image_size=image_size,
                device=CFG.device,
                batch_size=CFG.batch_size,
            )

            metrics, _, _, _ = calculate_classification_metrics(
                predictions,
                CLASS_ORDER,
            )

            resolution_records.append({
                **training_info,
                **{f"val_{k}": v for k, v in metrics.items()},
            })

    resolution_df = pd.DataFrame(resolution_records)
    resolution_df.to_csv(
        RESULTS_DIR / "resolution_ablation.csv",
        index=False,
    )

    resolution_summary = (
        resolution_df.groupby("image_size", observed=True)
        .agg(
            macro_f1_mean=("val_macro_f1", "mean"),
            macro_f1_std=("val_macro_f1", "std"),
            balanced_accuracy_mean=("val_balanced_accuracy", "mean"),
            balanced_accuracy_std=("val_balanced_accuracy", "std"),
            training_seconds_mean=("training_seconds", "mean"),
        )
        .reset_index()
    )
    display(resolution_summary)
else:
    print("Ablação de resolução desabilitada.")

## 21. Ablação de aumento de dados


In [ ]:
augmentation_records = []

if CFG.run_augmentation_ablation:
    for augmentation_name in ["no_aug", "physical_aug"]:
        for seed in SEEDS:
            run_name = (
                f"augmentation_{augmentation_name}_epochs{BEST_EPOCH_BUDGET}_"
                f"seed{seed}_img{BEST_IMAGE_SIZE}"
            )

            training_info = train_yolo_classifier(
                dataset_dir=IMAGE_DATASET_DIR,
                run_name=run_name,
                epochs=BEST_EPOCH_BUDGET,
                seed=seed,
                image_size=BEST_IMAGE_SIZE,
                augmentation_name=augmentation_name,
                project_dir=YOLO_RUNS_DIR / "augmentation_ablation",
            )

            predictions = predict_image_folder(
                model_path=training_info["best_model_path"],
                split_dir=IMAGE_DATASET_DIR / "val",
                image_size=BEST_IMAGE_SIZE,
                device=CFG.device,
                batch_size=CFG.batch_size,
            )

            metrics, _, _, _ = calculate_classification_metrics(
                predictions,
                CLASS_ORDER,
            )

            augmentation_records.append({
                **training_info,
                **{f"val_{k}": v for k, v in metrics.items()},
            })

    augmentation_df = pd.DataFrame(augmentation_records)
    augmentation_df.to_csv(
        RESULTS_DIR / "augmentation_ablation.csv",
        index=False,
    )

    augmentation_summary = (
        augmentation_df.groupby("augmentation", observed=True)
        .agg(
            macro_f1_mean=("val_macro_f1", "mean"),
            macro_f1_std=("val_macro_f1", "std"),
            balanced_accuracy_mean=("val_balanced_accuracy", "mean"),
            balanced_accuracy_std=("val_balanced_accuracy", "std"),
            training_seconds_mean=("training_seconds", "mean"),
        )
        .reset_index()
    )
    display(augmentation_summary)
else:
    print("Ablação de aumento de dados desabilitada.")

## 22. Análise qualitativa das predições

São selecionados exemplos de acertos, erros e casos de baixa confiança, com classe verdadeira, classe predita, confiança, redshift e classe original do Galaxy10.


In [ ]:
def select_qualitative_cases(
    predictions: pd.DataFrame,
    n_per_group: int = 6,
) -> dict[str, pd.DataFrame]:
    correct = predictions["true_class"] == predictions["predicted_class"]

    groups = {
        "correct_high_confidence": predictions.loc[correct].nlargest(
            n_per_group, "confidence"
        ),
        "correct_low_confidence": predictions.loc[correct].nsmallest(
            n_per_group, "confidence"
        ),
        "error_high_confidence": predictions.loc[~correct].nlargest(
            n_per_group, "confidence"
        ),
        "error_low_confidence": predictions.loc[~correct].nsmallest(
            n_per_group, "confidence"
        ),
    }
    return groups


def plot_case_grid(
    dataframe: pd.DataFrame,
    title: str,
    output_path: Path,
    columns: int = 3,
) -> None:
    if dataframe.empty:
        print("Nenhum caso disponível para:", title)
        return

    rows = math.ceil(len(dataframe) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    axes = np.atleast_1d(axes).reshape(rows, columns)

    for ax in axes.flat:
        ax.axis("off")

    for ax, row in zip(axes.flat, dataframe.itertuples(index=False)):
        image = Image.open(row.image_path)
        ax.imshow(image)
        redshift_text = "NaN" if pd.isna(row.redshift) else f"{row.redshift:.4f}"
        ax.set_title(
            f"True: {row.true_class}\n"
            f"Pred: {row.predicted_class} | p={row.confidence:.3f}\n"
            f"z={redshift_text} | original={int(row.original_class_id)}",
            fontsize=9,
        )
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.show()


qualitative_groups = select_qualitative_cases(test_predictions, n_per_group=6)

for group_name, group_df in qualitative_groups.items():
    group_df.to_csv(
        RESULTS_DIR / "best_yolo_test" / f"{group_name}.csv",
        index=False,
    )
    plot_case_grid(
        group_df,
        title=group_name.replace("_", " ").title(),
        output_path=FIGURES_DIR / f"{group_name}.png",
    )

## 23. Modelos de referência

ResNet18, EfficientNet-B0 e ViT-Tiny são avaliados nos mesmos *splits*, com inicialização pré-treinada em ImageNet e camada de saída adaptada às três macroclasses.


In [ ]:
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

try:
    import timm
except ImportError as exc:
    raise ImportError("Instale timm para executar o ViT-Tiny.") from exc


def make_torchvision_loaders(
    dataset_root: Path,
    image_size: int,
    batch_size: int,
    workers: int,
) -> tuple[DataLoader, DataLoader, DataLoader, list[str]]:
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=180),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])

    evaluation_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])

    train_dataset = datasets.ImageFolder(
        dataset_root / "train",
        transform=train_transform,
    )
    val_dataset = datasets.ImageFolder(
        dataset_root / "val",
        transform=evaluation_transform,
    )
    test_dataset = datasets.ImageFolder(
        dataset_root / "test",
        transform=evaluation_transform,
    )

    expected_classes = sorted(CLASS_ORDER)
    if train_dataset.classes != expected_classes:
        raise ValueError(
            f"Ordem inesperada das classes: {train_dataset.classes}. "
            f"Esperado: {expected_classes}."
        )

    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=workers,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=pin_memory,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
        train_dataset.classes,
    )


def build_baseline_model(
    model_name: str,
    n_classes: int,
) -> nn.Module:
    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, n_classes)
        return model

    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            n_classes,
        )
        return model

    if model_name == "vit_tiny_patch16_224":
        return timm.create_model(
            "vit_tiny_patch16_224",
            pretrained=True,
            num_classes=n_classes,
        )

    raise KeyError(f"Modelo baseline desconhecido: {model_name}")


def evaluate_torch_model(
    model: nn.Module,
    loader: DataLoader,
    class_names: list[str],
    device: torch.device,
) -> tuple[pd.DataFrame, float]:
    model.eval()
    records = []
    losses = []
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, targets)
            losses.append(float(loss.item()))

            probabilities = torch.softmax(logits, dim=1)
            confidence, predicted = probabilities.max(dim=1)

            for true_idx, pred_idx, conf in zip(
                targets.cpu().numpy(),
                predicted.cpu().numpy(),
                confidence.cpu().numpy(),
            ):
                records.append({
                    "true_class": class_names[int(true_idx)],
                    "predicted_class": class_names[int(pred_idx)],
                    "confidence": float(conf),
                })

    return pd.DataFrame(records), float(np.mean(losses))


def train_baseline_model(
    model_name: str,
    dataset_root: Path,
    image_size: int,
    epochs: int,
    seed: int,
    output_dir: Path,
) -> dict[str, Any]:
    set_global_seed(seed)

    train_loader, val_loader, test_loader, class_names = make_torchvision_loaders(
        dataset_root,
        image_size=image_size,
        batch_size=CFG.batch_size,
        workers=CFG.workers,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_baseline_model(model_name, len(class_names)).to(device)

    optimizer = AdamW(
        model.parameters(),
        lr=CFG.initial_lr,
        weight_decay=CFG.weight_decay,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    criterion = nn.CrossEntropyLoss()

    run_dir = output_dir / f"{model_name}_epochs{epochs}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "best.pt"

    best_macro_f1 = -np.inf
    best_epoch = -1
    epochs_without_improvement = 0
    history = []

    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        training_losses = []

        for images, targets in train_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
            training_losses.append(float(loss.item()))

        scheduler.step()

        val_predictions, val_loss = evaluate_torch_model(
            model,
            val_loader,
            class_names,
            device,
        )
        val_metrics, _, _, _ = calculate_classification_metrics(
            val_predictions,
            class_names,
        )

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(training_losses)),
            "val_loss": val_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })

        current_macro_f1 = float(val_metrics["macro_f1"])
        if current_macro_f1 > best_macro_f1:
            best_macro_f1 = current_macro_f1
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "model_name": model_name,
                "image_size": image_size,
                "seed": seed,
                "epoch": epoch,
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1

        print(
            f"{model_name} | seed={seed} | epoch={epoch}/{epochs} | "
            f"train_loss={np.mean(training_losses):.4f} | "
            f"val_loss={val_loss:.4f} | macro_f1={current_macro_f1:.4f}"
        )

        if epochs_without_improvement >= CFG.patience:
            print("Early stopping.")
            break

    elapsed = time.perf_counter() - started
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    test_predictions, test_loss = evaluate_torch_model(
        model,
        test_loader,
        class_names,
        device,
    )
    test_metrics, _, _, _ = calculate_classification_metrics(
        test_predictions,
        class_names,
    )

    test_predictions.to_csv(run_dir / "test_predictions.csv", index=False)

    return {
        "model_name": model_name,
        "seed": seed,
        "epochs_budget": epochs,
        "best_epoch": best_epoch,
        "image_size": image_size,
        "training_seconds": elapsed,
        "test_loss": test_loss,
        **{f"test_{k}": v for k, v in test_metrics.items()},
        "checkpoint_path": str(checkpoint_path),
    }

In [ ]:
baseline_records = []

if CFG.run_baselines:
    baseline_specs = {
        "resnet18": 224,
        "efficientnet_b0": 224,
        "vit_tiny_patch16_224": 224,
    }

    for model_name, image_size in baseline_specs.items():
        for seed in SEEDS:
            record = train_baseline_model(
                model_name=model_name,
                dataset_root=IMAGE_DATASET_DIR,
                image_size=image_size,
                epochs=BEST_EPOCH_BUDGET,
                seed=seed,
                output_dir=BASELINE_RUNS_DIR,
            )
            baseline_records.append(record)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    baseline_df = pd.DataFrame(baseline_records)
    baseline_df.to_csv(
        RESULTS_DIR / "baseline_models_all_runs.csv",
        index=False,
    )

    baseline_summary = (
        baseline_df.groupby("model_name", observed=True)
        .agg(
            runs=("seed", "count"),
            macro_f1_mean=("test_macro_f1", "mean"),
            macro_f1_std=("test_macro_f1", "std"),
            balanced_accuracy_mean=("test_balanced_accuracy", "mean"),
            balanced_accuracy_std=("test_balanced_accuracy", "std"),
            accuracy_mean=("test_accuracy", "mean"),
            accuracy_std=("test_accuracy", "std"),
            training_seconds_mean=("training_seconds", "mean"),
        )
        .reset_index()
    )

    display(baseline_summary)
else:
    print("Modelos baseline desabilitados.")

## 24. Comparação consolidada dos modelos


In [ ]:
final_summary_records = []

# YOLO selecionado
final_summary_records.append({
    "model": "YOLOv8n-CLS",
    "configuration": (
        f"epochs_budget={BEST_EPOCH_BUDGET}; seed={BEST_SEED}; "
        f"imgsz={BEST_IMAGE_SIZE}"
    ),
    **test_metrics,
})

# Baselines, quando executados
baseline_results_path = RESULTS_DIR / "baseline_models_all_runs.csv"
if baseline_results_path.exists():
    baseline_df = pd.read_csv(baseline_results_path)
    for model_name, group in baseline_df.groupby("model_name", observed=True):
        final_summary_records.append({
            "model": model_name,
            "configuration": f"{len(group)} seeds; imgsz={int(group['image_size'].iloc[0])}",
            "accuracy": group["test_accuracy"].mean(),
            "balanced_accuracy": group["test_balanced_accuracy"].mean(),
            "macro_f1": group["test_macro_f1"].mean(),
            "weighted_f1": group["test_weighted_f1"].mean(),
            "n_objects": group["test_n_objects"].iloc[0],
            "mean_confidence": group["test_mean_confidence"].mean(),
        })

final_model_table = pd.DataFrame(final_summary_records)
final_model_table.to_csv(
    RESULTS_DIR / "final_model_comparison_for_article.csv",
    index=False,
)

display(final_model_table)

## 25. Registro de execução e artefatos produzidos


In [ ]:
run_report = {
    "dataset": {
        "path": str(DATASET_PATH.resolve()),
        "md5": file_md5(DATASET_PATH),
        "n_original": int(len(catalog)),
        "n_balanced": int(len(balanced_manifest)),
        "class_counts_balanced": (
            balanced_manifest["macroclass"]
            .value_counts()
            .reindex(CLASS_ORDER)
            .astype(int)
            .to_dict()
        ),
        "split_counts": (
            balanced_manifest["split"].value_counts().astype(int).to_dict()
        ),
    },
    "taxonomy": {
        "original_class_names": ORIGINAL_CLASS_NAMES,
        "macroclass_map": MACROCLASS_MAP,
        "class_order": CLASS_ORDER,
    },
    "redshift_boundaries": redshift_config,
    "best_yolo_run": {
        "model_path": str(BEST_MODEL_PATH),
        "epochs_budget": BEST_EPOCH_BUDGET,
        "seed": BEST_SEED,
        "image_size": BEST_IMAGE_SIZE,
        "test_metrics": test_metrics,
    },
    "software": {
        "python": sys.version,
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "sklearn": sklearn.__version__,
        "platform": platform.platform(),
    },
    "configuration": {
        **asdict(CFG),
        "base_dir": str(CFG.base_dir),
    },
}

report_path = RESULTS_DIR / "complete_run_report.json"
with report_path.open("w", encoding="utf-8") as handle:
    json.dump(run_report, handle, indent=2, default=str)

print("Relatório final:", report_path)
print("\nArquivos principais:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(BASE_DIR))

## Notas de reprodutibilidade

- O arquivo público `Galaxy10_DECals_NoDuplicated.h5` é utilizado diretamente, sem *cross-match* adicional ou filtros externos de qualidade.
- Os rótulos são obtidos do campo `ans` e agregados segundo a taxonomia definida neste notebook.
- O balanceamento por *undersampling* produz uma amostra experimental balanceada; suas proporções não representam abundâncias cosmológicas.
- Os limites de redshift são estimados apenas no conjunto de treinamento, e a configuração principal do YOLO é selecionada pelo conjunto de validação.
- Sementes, hiperparâmetros, versões de software, integridade do conjunto de dados e artefatos de saída são registrados para auditoria da execução.
